In [ ]:
import os
import base64
import json
from dotenv import load_dotenv
import anthropic

# 1. .env dosyasindan API anahtarini yukle
load_dotenv(dotenv_path='../.env')
api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    print("Hata: API anahtari bulunamadi. .env dosyani kontrol et.")
else:
    print("API anahtari basariyla yuklendi.")

# 2. Anthropic istemcisini (client) baslat
client = anthropic.Anthropic(api_key=api_key)

# 3. Gorseli Base64 formatina ceviren fonksiyon
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# 4. Test belgesinin yolu (Gun 1 ile ayni belge)
image_path = "../data/raw_docs/test_talep_01.png"

try:
    base64_image = encode_image(image_path)
    print("Gorsel basariyla Base64 formatina cevrildi.")
except FileNotFoundError:
    print(f"Hata: Gorsel bulunamadi. Lutfen {image_path} yolunu kontrol et.")


In [ ]:
SYSTEM_PROMPT = """Sen bir dokuman analiz asistanisin. Sana resmi bir talep formunun gorseli verilecek.

Gorevin, belgedeki bilgileri SADECE asagidaki JSON semasina uygun sekilde cikarmaktir:

{
  "talep_eden": string,
  "tarih": string,        // YYYY-MM-DD formatinda normalize edilmis tarih
  "departman": string,
  "konu": string,
  "aciklama": string
}

KURALLAR:
- Yanitin SADECE gecerli bir JSON nesnesi olmali.
- Markdown kod blogu (uc backtick), aciklama cumlesi veya baska hicbir metin EKLEME. Yanitin '{' ile baslayip '}' ile bitmeli.
- Semadaki tum alanlari doldur. Bir bilgi belgede yoksa degerini null yap.
- Belgede olmayan bilgi UYDURMA.
"""

USER_INSTRUCTION = "Bu belgeyi yukaridaki semaya gore analiz et ve JSON olarak dondur."

print("Claude'a istek gonderiliyor, lutfen bekle...")
response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=1024,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": base64_image,
                    },
                },
                {
                    "type": "text",
                    "text": USER_INSTRUCTION
                }
            ],
        }
    ],
)

raw_text = "".join(block.text for block in response.content if block.type == "text")
print()
print("--- Claude'un Ham Yaniti ---")
print(raw_text)


In [ ]:
def extract_json(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()
    return json.loads(cleaned)

try:
    data = extract_json(raw_text)
    print("JSON parse basarili!")
    print()
    print(json.dumps(data, ensure_ascii=False, indent=2))

    expected_fields = {"talep_eden", "tarih", "departman", "konu", "aciklama"}
    missing = expected_fields - data.keys()
    if missing:
        print()
        print(f"Uyari: Eksik alanlar: {missing}")
    else:
        print()
        print("Tum beklenen alanlar mevcut.")
except json.JSONDecodeError as e:
    print(f"JSON parse hatasi: {e}")
    print("Ham yanit:")
    print(raw_text)
